
## 一、不平衡分类背景
### 1. 什么是不平衡数据？
- 正负样本数量差异悬殊，例如欺诈场景中：正常交易（负类0）占98%，欺诈交易（正类1）仅占2%。
- **问题**：直接用准确率（Accuracy）评估毫无意义——模型全预测为负类也能得到98%的准确率，但完全无法识别欺诈。

### 2. 核心解决思路
- **数据层面**：通过采样方法平衡训练集分布（测试集必须保持原始分布，保证评估真实）。
- **算法层面**：使用更关注少数类的评估指标（召回率、F1-score、AUC等），或调整模型权重。

### 3. 数据处理流程
```
原始数据 → 拆分训练集/测试集 → 仅对训练集做采样平衡 → 建模 → 用原始分布的测试集评估
```
- **关键原则**：**测试集绝不采样**，保持与真实业务一致的分布。


## 二、欠采样法（减少多数类）
### 1. 随机欠采样法
- **原理**：随机剔除部分多数类样本，使正负类数量接近。
- **优点**：简单高效，快速平衡数据。
- **缺点**：丢失大量多数类信息，可能导致模型泛化能力下降。

### 2. Tomek Link法
- **原理**：识别不同类别间最近的样本对（Tomek Link），删除其中的多数类样本，清洗分类边界。
- **优点**：保留更多有效信息，适合数据清洗。
- **缺点**：单独使用难以完全平衡数据，常与过采样结合。

---


## 三、过采样法（增加少数类）
### 1. 随机过采样法
- **原理**：随机重复抽取少数类样本，直到与多数类数量相当。
- **优点**：实现简单，不丢失信息。
- **缺点**：样本重复易导致**过拟合**，模型泛化能力差。

### 2. SMOTE法（合成少数类过采样）
- **原理**：在少数类样本及其近邻之间，通过插值人工合成新样本。
- **优点**：避免重复样本，一定程度上缓解过拟合。
- **缺点**：当样本分布零散时，易出现“样本入侵”（合成点混入多数类区域），仍可能过拟合。

---



## 四、综合采样法（过采样+欠采样）
- **代表算法**：SMOTE+Tomek Link（SMOTETomek）
- **流程**：
  1.  先用SMOTE合成少数类样本，扩充数据集。
  2.  再用Tomek Link剔除边界噪声样本，清洗“入侵”的合成样本。
- **优点**：兼顾数据平衡与模型泛化，是实践中更稳健的选择。


## 五、Python代码实战

### 1.环境与数据准备

In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv('imb_train.csv')
test = pd.read_csv('imb_test.csv')

print(f'训练集数据长度：{len(train)},测试集数据长度:{len(test)}')
train.sample(3)

训练集数据长度：14000,测试集数据长度:6000


,X1,X2,X3,X4,X5,cls
4369,0.277859,-0.490045,-0.252726,-0.451000,-0.557585,0
3155,-0.816515,0.790522,-2.300102,-1.471768,-0.923787,0
6792,-0.988669,0.847883,0.680777,-2.252814,-1.549786,0


In [2]:
# 使用collections库里面的Counter函数来实现统计分类
from collections import Counter
print('训练集中因变量cls分类情况：{}'.format(Counter(train['cls'])))
print('测试集因变量cls分类情况:{}'.format(Counter(test['cls'])))

训练集中因变量cls分类情况：Counter({0: 13644, 1: 356})
测试集因变量cls分类情况:Counter({0: 5848, 1: 152})


### 2.采样方式的实现

In [3]:
# 拆分自变量和因变量
y_train = train['cls'];y_test = test['cls']
X_train = train.loc[:,:'X5'];X_test = test.loc[:,:'X5']
print('不经过任何采样的原始数据y_train中的分类情况：{}'.format(Counter(y_train)))

from imblearn.over_sampling import RandomOverSampler
#采样策略sampling_strategy = 'auto'的auto默认抽成1:1
#如果想要另外的比例如1:5,甚至底线1:10，需要根据文档自行调整参数
# 先定义好模型，为开始正式训练拟合
ros = RandomOverSampler(random_state=0,sampling_strategy='auto')
X_ros,y_ros = ros.fit_resample(X=X_train,y=y_train)
print('随机过采样后，训练集y_ros中的分类情况:{}'.format(Counter(y_ros)))

# 同理，SMOTE的步骤也是如此
from imblearn.over_sampling import SMOTE
sos = SMOTE(random_state=0)
X_sos,y_sos = sos.fit_resample(X=X_train,y=y_train)
print('SMOTE过采样后，训练集y_sos中的分类情况：{}'.format(Counter(y_sos)))

# 同理，综合采样（先过采样再欠采样）
#combine表示组合抽样，所以SMOTE与Tomek这两个英文单词写在一起了
from imblearn.combine import SMOTETomek
kos = SMOTETomek(random_state=0)# 综合采样
X_kos,y_kos = kos.fit_resample(X=X_train,y=y_train)
print('综合采样后，训练集y_kos中的分类情况：{}'.format(Counter(y_kos)))

不经过任何采样的原始数据y_train中的分类情况：Counter({0: 13644, 1: 356})
随机过采样后，训练集y_ros中的分类情况:Counter({0: 13644, 1: 13644})
SMOTE过采样后，训练集y_sos中的分类情况：Counter({0: 13644, 1: 13644})
综合采样后，训练集y_kos中的分类情况：Counter({0: 13395, 1: 13395})


### 3. 决策树建模与评估

In [5]:
# 导入所需库
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics
from sklearn.model_selection import GridSearchCV

# 定义决策树
clf = DecisionTreeClassifier(criterion='gini', random_state=1234)
# 梯度优化的参数组合
param_grid = {'max_depth':[3,4,5,6],'max_leaf_nodes':[4,6,8,10,12]}
cv = GridSearchCV(clf, param_grid = param_grid, scoring='f1')

In [7]:
# X_train,y_train是没有经过任何操作的原始数据
# 第二组ros为随机过采样，第三组sos为SMOTE过采样
# 最后一组kos则为综合采样

data = [[X_train,y_train],
        [X_ros,y_ros],
        [X_sos,y_sos],
        [X_kos,y_kos]]

for features,labels in data:
    cv.fit(features,labels)# 对四组数据分别做模型
    # 使用·比例优良的（1:1~1:10）训练集来训练模型，用残酷的（分类为1的仅有2%）验证集来考验模型
    predict_test = cv.predict(X_test)
    print('auc:%.3f'%metrics.roc_auc_score(y_test,predict_test),
          'recall:%.3f'%metrics.recall_score(y_test,predict_test),
          'precision:%.3f'%metrics.precision_score(y_test,predict_test))

auc:0.747 recall:0.493 precision:0.987
auc:0.824 recall:0.783 precision:0.132
auc:0.819 recall:0.757 precision:0.143
auc:0.819 recall:0.757 precision:0.142



### 4. 权重法（算法层面解决不平衡）
- **原理**：通过`class_weight`参数，给少数类（正类）分配更高权重，让模型更关注分类错误的少数类样本。

In [8]:
# 采用改变样本权重的方法
param_grid2 = {
    'max_depth':[3,4,5,6],
    'max_leaf_nodes':[4,6,8,10,12],
    'class_weight':[{0:1,1:5},{0:1,1:10},{0:1,1:15}]
}
# clf是已经定义好的决策树
cv2 = GridSearchCV(clf, param_grid = param_grid2, scoring='f1')

In [9]:
# 还是需要在原始的数据集上使用权重法
cv2.fit(X_train,y_train)
predict_test2 = cv2.predict(X_test)

print(
    'auc:%.3f'%metrics.roc_auc_score(y_test,predict_test2),
    'recall:%.3f'%metrics.recall_score(y_test,predict_test2),
    'precision:%.3f'%metrics.precision_score(y_test,predict_test2)
)
cv2.best_params_

auc:0.806 recall:0.618 precision:0.740


{'class_weight': {0: 1, 1: 5}, 'max_depth': 4, 'max_leaf_nodes': 12}


## 六、结果分析与业务选择
### 1. 采样法结果对比
| 数据集       | AUC   | 召回率(Recall) | 精确率(Precision) | 特点                     |
|--------------|-------|----------------|-------------------|--------------------------|
| 原始数据     | 0.747 | 0.493          | 0.987             | 精确率高，漏检多（欺诈识别差） |
| 随机过采样   | 0.824 | 0.783          | 0.132             | 召回率高，误报多（大量正常交易被误判） |
| SMOTE过采样  | 0.819 | 0.757          | 0.143             | 略优于随机过采样         |
| 综合采样     | 0.819 | 0.757          | 0.142             | 更稳健，泛化性更好       |
| **权重法**   | 0.806 | 0.618          | 0.740             | **平衡召回率与精确率**，实践首选 |

### 2. 业务决策
- **高召回场景**（如医疗诊断、反欺诈）：优先选择**过采样法**，宁可误报也不漏检。
- **高精确场景**（如股票预测、广告投放）：优先选择**原始数据或权重法**，避免误报带来的损失。
- **平衡场景**：优先选择**权重法**或**综合采样法**，兼顾业务效率与模型效果。




## 七、核心总结
1.  **不平衡分类的本质**：不是简单让数据分布均衡，而是让模型**更关注少数类的识别效果**。
2.  **评估指标**：绝对不要用准确率，必须用**召回率、F1-score、AUC**等指标。
3.  **实践路径**：
    - 优先尝试**权重法**（简单高效，不改变数据分布）。
    - 若效果不佳，再尝试**综合采样法**（SMOTETomek）。
    - 最后根据业务需求，在召回率和精确率之间做 trade-off。